In [43]:
# =====================================================
# Feature Engineering - Rider-Level Churn Dataset
# =====================================================

import pandas as pd
import numpy as np

In [44]:
# =====================================================
# Load Cleaned Datasets
# =====================================================

riders = pd.read_csv("../data/processed/riders_clean.csv")
drivers = pd.read_csv("../data/processed/drivers_clean.csv")
trips = pd.read_csv("../data/processed/trips_clean.csv")
sessions = pd.read_csv("../data/processed/sessions_clean.csv")

In [45]:
# =====================================================
# Convert Date Columns
# =====================================================

trips["pickup_time"] = pd.to_datetime(trips["pickup_time"], errors="coerce", utc=True)
trips["dropoff_time"] = pd.to_datetime(trips["dropoff_time"], errors="coerce", utc=True)

riders["signup_date"] = pd.to_datetime(riders["signup_date"], errors="coerce", utc=True)

drivers["signup_date"] = pd.to_datetime(drivers["signup_date"], errors="coerce", utc=True)
drivers["last_active"] = pd.to_datetime(drivers["last_active"], errors="coerce", utc=True)

sessions["session_time"] = pd.to_datetime(sessions["session_time"], errors="coerce", utc=True)

In [46]:
# =====================================================
# Snapshot Date
# =====================================================

snapshot_date = trips["pickup_time"].max()

print("Snapshot date:", snapshot_date)

Snapshot date: 2025-04-27 23:43:26+00:00


In [47]:
# =====================================================
# Merge Trips with Driver Information
# =====================================================

trips_drivers = trips.merge(
    drivers[
        [
            "driver_id",
            "rating",
            "vehicle_type",
            "acceptance_rate"
        ]
    ],
    on="driver_id",
    how="left"
)

# Flag trips where driver details are missing
trips_drivers["driver_info_missing"] = (
    trips_drivers["rating"].isna()
).astype(int)

In [48]:
# =====================================================
# Trip-Level Features Before Aggregation
# =====================================================

trips_drivers["fare_per_minute"] = (
    trips_drivers["fare"] / trips_drivers["trip_duration_minutes"]
).replace([np.inf, -np.inf], 0).fillna(0)

trips_drivers["tip_percentage"] = (
    trips_drivers["tip"] / trips_drivers["fare"] * 100
).replace([np.inf, -np.inf], 0).fillna(0)

trips_drivers["pickup_hour"] = trips_drivers["pickup_time"].dt.hour

trips_drivers["is_weekend"] = (
    trips_drivers["pickup_time"].dt.dayofweek >= 5
).astype(int)

trips_drivers["peak_hour_trip"] = (
    trips_drivers["pickup_hour"].between(7, 9) |
    trips_drivers["pickup_hour"].between(16, 19)
).astype(int)

trips_drivers["night_trip"] = (
    (trips_drivers["pickup_hour"] >= 22) |
    (trips_drivers["pickup_hour"] <= 5)
).astype(int)

trips_drivers["surge_trip"] = (
    trips_drivers["surge_multiplier"] > 1
).astype(int)

In [49]:
# =====================================================
# Aggregate Trip Features Per Rider
# =====================================================

trip_features = (
    trips_drivers
    .groupby("user_id")
    .agg(
        total_trips=("trip_id", "nunique"),
        avg_fare=("fare", "mean"),
        total_fare=("fare", "sum"),
        avg_surge=("surge_multiplier", "mean"),
        avg_tip=("tip", "mean"),
        avg_tip_percentage=("tip_percentage", "mean"),
        avg_trip_duration=("trip_duration_minutes", "mean"),
        avg_fare_per_minute=("fare_per_minute", "mean"),
        avg_driver_rating=("rating", "mean"),
        avg_driver_acceptance=("acceptance_rate", "mean"),
        driver_info_missing_rate=("driver_info_missing", "mean"),
        weekend_trip_rate=("is_weekend", "mean"),
        peak_hour_trip_rate=("peak_hour_trip", "mean"),
        night_trip_rate=("night_trip", "mean"),
        surge_trip_rate=("surge_trip", "mean"),
        most_recent_trip=("pickup_time", "max"),
        first_trip=("pickup_time", "min"),
        favourite_payment_type=("payment_type", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
        favourite_weather=("weather", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
        favourite_vehicle_type=("vehicle_type", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
    )
    .reset_index()
)

display(trip_features.head())

,user_id,total_trips,avg_fare,total_fare,avg_surge,avg_tip,avg_tip_percentage,avg_trip_duration,avg_fare_per_minute,avg_driver_rating,...,driver_info_missing_rate,weekend_trip_rate,peak_hour_trip_rate,night_trip_rate,surge_trip_rate,most_recent_trip,first_trip,favourite_payment_type,favourite_weather,favourite_vehicle_type
0,R00000,25,14.642000,366.05,1.096000,0.161200,1.170551,30.320000,0.734031,4.295833,...,0.040000,0.160000,0.200000,0.360000,0.280000,2025-04-02 14:46:29+00:00,2024-05-01 07:21:52+00:00,Mobile Money,Sunny,Sedan
1,R00001,14,12.895000,180.53,1.071429,0.054286,0.368973,28.642857,0.807591,4.135714,...,0.000000,0.285714,0.214286,0.285714,0.214286,2025-04-22 04:35:17+00:00,2024-05-10 18:14:41+00:00,Card,Sunny,Sedan
2,R00002,24,15.791250,378.99,1.191667,0.217083,1.831782,31.541667,0.586718,4.150000,...,0.000000,0.125000,0.416667,0.166667,0.375000,2025-04-13 00:08:00+00:00,2024-06-18 17:48:24+00:00,Card,Sunny,Sedan
3,R00003,9,13.496667,121.47,1.155556,0.096667,0.846863,32.555556,0.489055,4.237500,...,0.111111,0.333333,0.222222,0.666667,0.333333,2025-02-25 04:22:32+00:00,2024-05-15 05:13:12+00:00,Card,Sunny,Suv
4,R00004,16,16.776875,268.43,1.262500,0.586250,5.785145,36.125000,0.633158,4.156250,...,0.000000,0.437500,0.312500,0.250000,0.437500,2025-04-15 05:30:04+00:00,2024-05-23 13:02:45+00:00,Mobile Money,Sunny,Sedan


In [50]:
# =====================================================
# Aggregate Session Features Per Rider
# =====================================================

session_features = (
    sessions
    .groupby("rider_id")
    .agg(
        total_sessions=("session_id", "nunique"),
        avg_time_on_app=("time_on_app", "mean"),
        avg_pages_visited=("pages_visited", "mean"),
        conversion_rate=("converted", "mean")
    )
    .reset_index()
    .rename(columns={"rider_id": "user_id"})
)

display(session_features.head())

,user_id,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate
0,R00000,4,92.000000,3.000000,0.25
1,R00001,3,174.666667,2.666667,0.00
2,R00002,3,191.000000,3.000000,0.00
3,R00003,3,75.333333,1.666667,0.00
4,R00004,2,17.000000,2.500000,0.00


In [51]:
# =====================================================
# Merge Rider + Trip + Session Features
# =====================================================

rider_level_df = riders.merge(
    trip_features,
    on="user_id",
    how="left"
)

rider_level_df = rider_level_df.merge(
    session_features,
    on="user_id",
    how="left"
)

print("Shape after merging:", rider_level_df.shape)
display(rider_level_df.head())

Shape after merging: (10000, 32)


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,is_referred,total_trips,avg_fare,...,surge_trip_rate,most_recent_trip,first_trip,favourite_payment_type,favourite_weather,favourite_vehicle_type,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate
0,R00000,2025-01-24 00:00:00+00:00,Bronze,34.729629,Nairobi,5.0,R00001,1,25,14.642000,...,0.280000,2025-04-02 14:46:29+00:00,2024-05-01 07:21:52+00:00,Mobile Money,Sunny,Sedan,4.0,92.000000,3.000000,0.25
1,R00001,2024-09-09 00:00:00+00:00,Bronze,34.571020,Nairobi,4.7,Unknown,0,14,12.895000,...,0.214286,2025-04-22 04:35:17+00:00,2024-05-10 18:14:41+00:00,Card,Sunny,Sedan,3.0,174.666667,2.666667,0.00
2,R00002,2024-09-07 00:00:00+00:00,Bronze,47.133960,Lagos,4.2,Unknown,0,24,15.791250,...,0.375000,2025-04-13 00:08:00+00:00,2024-06-18 17:48:24+00:00,Card,Sunny,Sedan,3.0,191.000000,3.000000,0.00
3,R00003,2025-03-17 00:00:00+00:00,Bronze,41.658628,Nairobi,4.9,Unknown,0,9,13.496667,...,0.333333,2025-02-25 04:22:32+00:00,2024-05-15 05:13:12+00:00,Card,Sunny,Suv,3.0,75.333333,1.666667,0.00
4,R00004,2024-08-20 00:00:00+00:00,Silver,40.681709,Lagos,3.9,R00002,1,16,16.776875,...,0.437500,2025-04-15 05:30:04+00:00,2024-05-23 13:02:45+00:00,Mobile Money,Sunny,Sedan,2.0,17.000000,2.500000,0.00


In [52]:
# =====================================================
# Fill Missing Values
# =====================================================

activity_cols = [
    "total_trips",
    "avg_fare",
    "total_fare",
    "avg_surge",
    "avg_tip",
    "avg_tip_percentage",
    "avg_trip_duration",
    "avg_fare_per_minute",
    "driver_info_missing_rate",
    "weekend_trip_rate",
    "peak_hour_trip_rate",
    "night_trip_rate",
    "surge_trip_rate",
    "total_sessions",
    "avg_time_on_app",
    "avg_pages_visited",
    "conversion_rate"
]

rider_level_df[activity_cols] = rider_level_df[activity_cols].fillna(0)

# Fill driver averages using median, not zero
rider_level_df["avg_driver_rating"] = rider_level_df["avg_driver_rating"].fillna(
    rider_level_df["avg_driver_rating"].median()
)

rider_level_df["avg_driver_acceptance"] = rider_level_df["avg_driver_acceptance"].fillna(
    rider_level_df["avg_driver_acceptance"].median()
)

categorical_fill_cols = [
    "favourite_payment_type",
    "favourite_weather",
    "favourite_vehicle_type"
]

rider_level_df[categorical_fill_cols] = (
    rider_level_df[categorical_fill_cols]
    .fillna("Unknown")
)

In [53]:
# =====================================================
# Customer Tenure and Recency Features
# =====================================================

rider_level_df["account_tenure_days"] = (
    snapshot_date - rider_level_df["signup_date"]
).dt.days

rider_level_df["days_since_last_trip"] = (
    snapshot_date - rider_level_df["most_recent_trip"]
).dt.days

rider_level_df["days_since_first_trip"] = (
    snapshot_date - rider_level_df["first_trip"]
).dt.days

rider_level_df["days_since_last_trip"] = (
    rider_level_df["days_since_last_trip"]
    .fillna(999)
)

rider_level_df["days_since_first_trip"] = (
    rider_level_df["days_since_first_trip"]
    .fillna(999)
)

rider_level_df["account_tenure_days"] = (
    rider_level_df["account_tenure_days"]
    .fillna(rider_level_df["account_tenure_days"].median())
)

In [54]:
# =====================================================
# Behavioural Features
# =====================================================

rider_level_df["trip_velocity"] = (
    rider_level_df["total_trips"] /
    np.clip(rider_level_df["account_tenure_days"], 1, None)
)

rider_level_df["trips_per_session"] = (
    rider_level_df["total_trips"] /
    np.clip(rider_level_df["total_sessions"], 1, None)
)

rider_level_df["engagement_score"] = (
    rider_level_df["total_sessions"] * 0.4 +
    rider_level_df["avg_time_on_app"] * 0.4 +
    rider_level_df["avg_pages_visited"] * 0.2
)

# Standardize referral values
rider_level_df["referred_by"] = (
    rider_level_df["referred_by"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Create referral flag
rider_level_df["is_referred"] = (
    rider_level_df["referred_by"] != "Unknown"
).astype(int)

In [55]:
# =====================================================
# Additional Behavioural Features
# =====================================================

# Average number of days between trips
rider_level_df["avg_days_between_trips"] = (
    rider_level_df["account_tenure_days"] /
    np.clip(rider_level_df["total_trips"], 1, None)
)

# Fare per app session
rider_level_df["fare_per_session"] = (
    rider_level_df["total_fare"] /
    np.clip(rider_level_df["total_sessions"], 1, None)
)

# Trips per day active
rider_level_df["trips_per_active_day"] = (
    rider_level_df["total_trips"] /
    np.clip(rider_level_df["days_since_first_trip"], 1, None)
)

# Driver quality score
rider_level_df["driver_quality_score"] = (
    rider_level_df["avg_driver_rating"] *
    rider_level_df["avg_driver_acceptance"]
)

# Tip ratio compared to fare
rider_level_df["tip_ratio"] = (
    rider_level_df["avg_tip"] /
    np.clip(rider_level_df["avg_fare"], 1, None)
)

# Spending intensity
rider_level_df["spend_per_day"] = (
    rider_level_df["total_fare"] /
    np.clip(rider_level_df["account_tenure_days"], 1, None)
)

# Session engagement intensity
rider_level_df["session_intensity"] = (
    rider_level_df["total_sessions"] /
    np.clip(rider_level_df["account_tenure_days"], 1, None)
)

# App usage quality score
rider_level_df["app_usage_score"] = (
    rider_level_df["avg_time_on_app"] *
    rider_level_df["avg_pages_visited"]
)

# Conversion-adjusted engagement
rider_level_df["conversion_engagement_score"] = (
    rider_level_df["engagement_score"] *
    rider_level_df["conversion_rate"]
)

# Weekend and night behaviour combined
rider_level_df["non_standard_trip_rate"] = (
    rider_level_df["weekend_trip_rate"] +
    rider_level_df["night_trip_rate"]
)

# Surge sensitivity proxy
rider_level_df["surge_fare_interaction"] = (
    rider_level_df["avg_surge"] *
    rider_level_df["avg_fare"]
)

# Loyalty spending interaction
rider_level_df["loyalty_spend_interaction"] = (
    rider_level_df["total_fare"] *
    rider_level_df["trip_velocity"]
)

In [56]:
# =====================================================
# Age Group
# =====================================================

rider_level_df["age_group"] = pd.cut(
    rider_level_df["age"],
    bins=[18, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

rider_level_df["age_group"] = (
    rider_level_df["age_group"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)

In [57]:
# =====================================================
# Target Variable
# Business Rule: Churned if no trip in the last 30 days
# =====================================================

rider_level_df["is_churned"] = (
    rider_level_df["days_since_last_trip"] > 30
).astype(int)

print("Target counts:")
print(rider_level_df["is_churned"].value_counts())

print("\nTarget percentage:")
print(rider_level_df["is_churned"].value_counts(normalize=True) * 100)

Target counts:
is_churned
0    8114
1    1886
Name: count, dtype: int64

Target percentage:
is_churned
0    81.14
1    18.86
Name: proportion, dtype: float64


In [58]:
# =====================================================
# Remove Leakage Columns
# =====================================================

leakage_cols = [
    "churn_prob",
    "most_recent_trip",
    "first_trip",
    "days_since_last_trip",
    "days_since_first_trip"
]

rider_level_df.drop(
    columns=leakage_cols,
    inplace=True,
    errors="ignore"
)

print("Leakage columns removed successfully.")

Leakage columns removed successfully.


In [59]:
# =====================================================
# Leakage Check
# =====================================================

existing_leakage_cols = [
    col for col in leakage_cols
    if col in rider_level_df.columns
]

print("Leakage columns still present:", existing_leakage_cols)

Leakage columns still present: []


In [60]:
# =====================================================
# Final Validation Checks
# =====================================================

print("=" * 60)
print("Feature Engineering Validation")
print("=" * 60)

print("Dataset shape:", rider_level_df.shape)

print("\nDuplicate columns:")
print(rider_level_df.columns[rider_level_df.columns.duplicated()])

print("\nDuplicate User IDs:")
print(rider_level_df["user_id"].duplicated().sum())

print("\nMissing values:")
missing = rider_level_df.isnull().sum()
print(missing[missing > 0])

print("\nTarget distribution:")
print(rider_level_df["is_churned"].value_counts())

assert rider_level_df["user_id"].is_unique, "There are duplicate user_id values."
assert rider_level_df.columns.duplicated().sum() == 0, "There are duplicate column names."
assert existing_leakage_cols == [], "Leakage columns are still present."

print("\nValidation passed.")
display(rider_level_df.head())

Feature Engineering Validation
Dataset shape: (10000, 48)

Duplicate columns:
Index([], dtype='str')

Duplicate User IDs:
0

Missing values:
Series([], dtype: int64)

Target distribution:
is_churned
0    8114
1    1886
Name: count, dtype: int64

Validation passed.


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,is_referred,total_trips,avg_fare,...,tip_ratio,spend_per_day,session_intensity,app_usage_score,conversion_engagement_score,non_standard_trip_rate,surge_fare_interaction,loyalty_spend_interaction,age_group,is_churned
0,R00000,2025-01-24 00:00:00+00:00,Bronze,34.729629,Nairobi,5.0,R00001,1,25,14.642000,...,0.011009,3.936022,0.043011,276.000000,9.75,0.520000,16.047632,98.400538,26-35,0
1,R00001,2024-09-09 00:00:00+00:00,Bronze,34.571020,Nairobi,4.7,Unknown,0,14,12.895000,...,0.004210,0.784913,0.013043,465.777778,0.00,0.571429,13.816071,10.988783,26-35,0
2,R00002,2024-09-07 00:00:00+00:00,Bronze,47.133960,Lagos,4.2,Unknown,0,24,15.791250,...,0.013747,1.633578,0.012931,573.000000,0.00,0.291667,18.817906,39.205862,46-60,0
3,R00003,2025-03-17 00:00:00+00:00,Bronze,41.658628,Nairobi,4.9,Unknown,0,9,13.496667,...,0.007162,2.962683,0.073171,125.555556,0.00,1.000000,15.596148,26.664146,36-45,1
4,R00004,2024-08-20 00:00:00+00:00,Silver,40.681709,Lagos,3.9,R00002,1,16,16.776875,...,0.034944,1.073720,0.008000,42.500000,0.00,0.687500,21.180805,17.179520,36-45,0


In [61]:
# =====================================================
# Ensure One Row Per Rider
# =====================================================

assert rider_level_df["user_id"].is_unique, (
    "There are duplicate user_id values."
)

print("✓ Validation passed: One row per rider.")

✓ Validation passed: One row per rider.


In [62]:
# =====================================================
# Ensure Dataset Integrity
# =====================================================

# One row per rider
assert rider_level_df["user_id"].is_unique, (
    "There are duplicate user_id values."
)

# No duplicate column names
assert rider_level_df.columns.duplicated().sum() == 0, (
    "Duplicate column names found."
)

# Target column must exist
assert "is_churned" in rider_level_df.columns, (
    "Target column 'is_churned' is missing."
)

print("✓ All validation checks passed.")

✓ All validation checks passed.


In [63]:
# =====================================================
# Save Rider-Level Modelling Dataset
# =====================================================

rider_level_df.to_csv(
    "../data/processed/rider_level_churn_dataset.csv",
    index=False
)

print("Rider-level churn dataset saved successfully.")
print("Final shape:", rider_level_df.shape)

Rider-level churn dataset saved successfully.
Final shape: (10000, 48)


In [64]:
print(rider_level_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   user_id                      10000 non-null  str                
 1   signup_date                  10000 non-null  datetime64[us, UTC]
 2   loyalty_status               10000 non-null  str                
 3   age                          10000 non-null  float64            
 4   city                         10000 non-null  str                
 5   avg_rating_given             10000 non-null  float64            
 6   referred_by                  10000 non-null  str                
 7   is_referred                  10000 non-null  int64              
 8   total_trips                  10000 non-null  int64              
 9   avg_fare                     10000 non-null  float64            
 10  total_fare                   10000 non-null  float64      